# LAB GenAI - LLMs - OpenAI GPT API Exercises

In [1]:
# pip install openai python-dotenv tiktoken
import math
from collections import Counter

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()            # reads OPENAI_API_KEY from a .env file in the project folder
client = OpenAI()        # picks the key up from the environment automatically

MODEL = "gpt-4o-mini"                     # cheap chat model; swap for whatever your course uses
LEGACY_MODEL = "gpt-3.5-turbo-instruct"   # only for the legacy /v1/completions endpoint

## 1. Basic Conversation
**Exercise:** Create a simple chatbot that can answer basic questions about a given topic (e.g., history, technology).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [2]:
SYSTEM = {
    "role": "system",
    "content": ("You are a friendly history tutor. Answer in 2-3 sentences. "
                "If the question is not about history, politely say so."),
}

def ask_history_bot(question, **params):
    """Single-turn question. Returns a list because n>1 gives several answers."""
    messages = [SYSTEM, {"role": "user", "content": question}]
    response = client.chat.completions.create(model=MODEL, messages=messages, **params)
    return [(c.message.content, c.finish_reason) for c in response.choices]


question = "Why did the Western Roman Empire fall?"

experiments = {
    "baseline":          dict(temperature=0.7),
    "temperature=0":     dict(temperature=0),
    "temperature=1.6":   dict(temperature=1.6),
    "max_tokens=20":     dict(max_tokens=20),
    "top_p=0.1":         dict(top_p=0.1),
    "frequency_pen=2":   dict(frequency_penalty=2.0),
    "presence_pen=2":    dict(presence_penalty=2.0),
    "n=3":               dict(n=3, temperature=1.0),
    "stop='.'":          dict(stop=["."]),
}

for name, params in experiments.items():
    print(f"===== {name} {params} =====")
    for i, (text, reason) in enumerate(ask_history_bot(question, **params), 1):
        print(f"[{i}] ({reason}) {text}\n")

===== baseline {'temperature': 0.7} =====
[1] (stop) The fall of the Western Roman Empire was due to a combination of internal weaknesses, such as political instability, economic troubles, and overreliance on mercenary forces, as well as external pressures from invasions by barbarian tribes like the Visigoths and Vandals. The traditional date for the fall is 476 AD, when the last Roman emperor, Romulus Augustulus, was deposed. This event marked the end of ancient Rome and the beginning of the Middle Ages in Europe.

===== temperature=0 {'temperature': 0} =====
[1] (stop) The fall of the Western Roman Empire was due to a combination of factors, including economic decline, military defeats, and political instability. Invasions by barbarian tribes, such as the Visigoths and Vandals, further weakened the empire, culminating in the deposition of the last emperor, Romulus Augustulus, in 476 AD. Additionally, internal strife and corruption contributed to its inability to effectively govern an

In [3]:
def run_chatbot():
    messages = [SYSTEM]                                   # the conversation "memory"
    while True:
        user = input("You: ")
        if user.strip().lower() in {"quit", "exit"}:
            break
        messages.append({"role": "user", "content": user})
        reply = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0.7
        ).choices[0].message.content
        messages.append({"role": "assistant", "content": reply})
        print("Bot:", reply)

# run_chatbot()

## 2. Summarization
**Exercise:** Write a script that takes a long text input and summarizes it into a few sentences.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [4]:
long_text = """Paste any long article here (a Wikipedia section works well).
The longer and more detailed, the easier it is to see parameter effects."""

def summarize(text, n_sentences=3, **params):
    messages = [
        {"role": "system", "content": "You are an expert editor who writes faithful, concise summaries. Never add facts that are not in the text."},
        {"role": "user", "content": f"Summarize the following text in {n_sentences} sentences:\n\n{text}"},
    ]
    return client.chat.completions.create(model=MODEL, messages=messages, **params)


for params in [dict(temperature=0), dict(temperature=1.2), dict(max_tokens=30),
               dict(top_p=0.2), dict(frequency_penalty=1.5), dict(presence_penalty=1.5)]:
    r = summarize(long_text, **params)
    print(f"--- {params} | finish_reason={r.choices[0].finish_reason} ---")
    print(r.choices[0].message.content, "\n")

--- {'temperature': 0} | finish_reason=stop ---
Please provide the text you would like summarized, and I will create a concise summary for you. 

--- {'temperature': 1.2} | finish_reason=stop ---
The text instructs users to paste a long article, such as a section from Wikipedia, to facilitate the observation of parameter effects. It suggests that comprehensive and detailed texts are beneficial for this purpose. The overall aim is to enable a clear analysis by using longer sections of content. 

--- {'max_tokens': 30} | finish_reason=length ---
The instructions suggest pasting a long article, such as a section from Wikipedia, to analyze the effects of parameters within it. Longer and more detailed texts 

--- {'top_p': 0.2} | finish_reason=stop ---
Please provide the text you would like summarized, and I will create a concise summary for you. 

--- {'frequency_penalty': 1.5} | finish_reason=stop ---
Please provide the text you would like summarized, and I will create a concise summary f

In [5]:
r = summarize(long_text, temperature=0, max_tokens=150, logprobs=True, top_logprobs=3)
print(r.choices[0].message.content, "\n")

token_info = r.choices[0].logprobs.content
for tok in token_info[:12]:
    alts = ", ".join(f"{a.token!r}: {math.exp(a.logprob):.1%}" for a in tok.top_logprobs)
    print(f"{tok.token!r:15} p={math.exp(tok.logprob):.1%} | top alternatives → {alts}")

avg_logprob = sum(t.logprob for t in token_info) / len(token_info)
print(f"\nAverage token probability (geometric mean): {math.exp(avg_logprob):.1%}")

Please provide the text you would like summarized, and I will create a concise summary for you. 

'Please'        p=60.5% | top alternatives → 'Please': 60.5%, 'The': 28.6%, "I'm": 2.3%
' provide'      p=99.5% | top alternatives → ' provide': 99.5%, ' paste': 0.5%, ' share': 0.0%
' the'          p=97.6% | top alternatives → ' the': 97.6%, ' a': 2.3%, ' an': 0.1%
' text'         p=90.3% | top alternatives → ' text': 90.3%, ' article': 5.1%, ' long': 2.7%
' you'          p=67.3% | top alternatives → ' you': 67.3%, " you'd": 17.0%, ' or': 15.0%
' would'        p=99.2% | top alternatives → ' would': 99.2%, ' want': 0.7%, '’d': 0.1%
' like'         p=100.0% | top alternatives → ' like': 100.0%, 'like': 0.0%, ' liked': 0.0%
' summarized'   p=61.8% | top alternatives → ' summarized': 61.8%, ' me': 37.5%, ' to': 0.8%
','             p=81.5% | top alternatives → ',': 81.5%, '.': 18.2%, ' so': 0.2%
' and'          p=99.5% | top alternatives → ' and': 99.5%, ' as': 0.5%, ' so': 0.0%
' I'         

In [15]:
def chat_best_of(messages, best_of=3, **params):
    """Generate `best_of` candidates and return the one with the highest
    average token log-probability (the same rule the legacy best_of used)."""
    r = client.chat.completions.create(
        model=MODEL, messages=messages, n=best_of, logprobs=True, **params
    )

    def avg_logprob(choice):
        lps = [t.logprob for t in choice.logprobs.content]
        return sum(lps) / len(lps)

    ranked = sorted(r.choices, key=avg_logprob, reverse=True)
    for c in ranked:
        print(f"avg p = {math.exp(avg_logprob(c)):.1%} | {c.message.content[:90]}...")
    return ranked[0].message.content


summary_messages = [
    {"role": "system", "content": "You are an expert editor who writes faithful, concise summaries."},
    {"role": "user", "content": f"Summarize the following text in 3 sentences:\n\n{long_text}"},
]

for k in (1, 4):
    print(f"\n===== best_of={k} =====")
    best = chat_best_of(summary_messages, best_of=k, temperature=0.9, max_tokens=150)
    print("\nCHOSEN:\n", best)


===== best_of=1 =====
avg p = 63.6% | Please provide the text or article you would like summarized, and I’ll create a concise su...

CHOSEN:
 Please provide the text or article you would like summarized, and I’ll create a concise summary for you.

===== best_of=4 =====
avg p = 82.0% | Please provide the text you would like summarized, and I will create a concise summary for...
avg p = 71.6% | Please provide the text you would like summarized, and I will be happy to help!...
avg p = 71.1% | Please provide the text you would like me to summarize, and I will be happy to assist you....
avg p = 47.1% | To summarize the provided text in three sentences, I need the actual content or article yo...

CHOSEN:
 Please provide the text you would like summarized, and I will create a concise summary for you.


## 3. Translation
**Exercise:** Develop a tool that translates text from one language to another using the API.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

In [7]:
def translate(text, target="Spanish", **params):
    messages = [
        {"role": "system", "content": f"You are a professional translator. Translate the user's text into {target}. Return ONLY the translation, with no explanations or quotes."},
        {"role": "user", "content": text},
    ]
    r = client.chat.completions.create(model=MODEL, messages=messages, **params)
    return r.choices[0].message.content

sentence = "Hello my friend, the meeting has been moved to Thursday because the manager is sick."

for params in [dict(temperature=0), dict(temperature=1.5), dict(max_tokens=8),
               dict(top_p=0.1), dict(frequency_penalty=2), dict(presence_penalty=2)]:
    print(f"{str(params):30} → {translate(sentence, **params)}")

{'temperature': 0}             → Hola mi amigo, la reunión se ha trasladado a jueves porque el gerente está enfermo.
{'temperature': 1.5}           → Hola amigo mío, la reunión se ha movido para el jueves porque el gerente está enfermo.
{'max_tokens': 8}              → Hola mi amigo, la reunión se ha
{'top_p': 0.1}                 → Hola mi amigo, la reunión se ha trasladado al jueves porque el gerente está enfermo.
{'frequency_penalty': 2}       → Hola mi amigo, la reunión se ha trasladado a jueves porque el gerente está enfermo.
{'presence_penalty': 2}        → Hola amigo, la reunión se ha trasladado a jueves porque el gerente está enfermo.


In [8]:
import tiktoken

enc = tiktoken.encoding_for_model(MODEL)

def token_ids(words):
    """Collect token IDs for a word in its common variants (with/without leading space, capitalised)."""
    ids = {}
    for w in words:
        for variant in (w, " " + w, w.capitalize(), " " + w.capitalize()):
            toks = enc.encode(variant)
            ids[variant] = toks
    return ids

ban = token_ids(["amigo"])
print(ban)   # check: is each variant ONE token or several?

bias = {tok_id: -100 for toks in ban.values() for tok_id in toks}
print("Normal: ", translate(sentence, temperature=0))
print("Banned: ", translate(sentence, temperature=0, logit_bias=bias))

{'amigo': [313, 6235], ' amigo': [39586], 'Amigo': [6275, 6235], ' Amigo': [4820, 6235]}
Normal:  Hola mi amigo, la reunión se ha trasladado a jueves porque el gerente está enfermo.
Banned:  Hola mi amiga, la reunión se ha trasladado a jueves porque el gerente está enfermo.


In [16]:
def translate_with_echo(text, target="French", **params):
    """Emulate echo=True: return the original prompt glued to the model's output."""
    translation = translate(text, target=target, **params)
    return f"English: {text}\n{target}: {translation}"

print(translate_with_echo("The weather is nice today.", temperature=0, stop=["\n"]))

English: The weather is nice today.
French: Le temps est agréable aujourd'hui.


## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool that determines the sentiment of a given text (positive, negative, neutral).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [10]:
LABELS = ["positive", "negative", "neutral"]

def classify_sentiment(text, **params):
    messages = [
        {"role": "system", "content": "Classify the sentiment of the user's text. Answer with exactly one lowercase word: positive, negative, or neutral."},
        {"role": "user", "content": text},
    ]
    return client.chat.completions.create(model=MODEL, messages=messages, **params)


def sentiment_with_confidence(text):
    """Label + probability of each label, read from the first token's logprobs."""
    r = classify_sentiment(text, temperature=0, max_tokens=2, logprobs=True, top_logprobs=5)
    first_token = r.choices[0].logprobs.content[0]
    probs = {lbl: 0.0 for lbl in LABELS}
    for alt in first_token.top_logprobs:
        tok = alt.token.strip().lower()
        if tok in probs:                      # assumes each label is a single token
            probs[tok] += math.exp(alt.logprob)
    return r.choices[0].message.content.strip().lower(), probs


def sentiment_vote(text, n=5, temperature=1.0):
    """Self-consistency: sample n answers and take the majority."""
    r = classify_sentiment(text, n=n, temperature=temperature, max_tokens=2)
    votes = Counter(c.message.content.strip().lower() for c in r.choices)
    label, count = votes.most_common(1)[0]
    return label, count / n, dict(votes)


reviews = [
    "Absolutely loved it, would buy again!",
    "The product arrived broken and support never answered.",
    "It's a phone. It makes calls.",
    "Great screen, but the battery dies by lunchtime.",   # mixed: the interesting case
]

rows = []
for rev in reviews:
    label, probs = sentiment_with_confidence(rev)
    vote_label, agreement, votes = sentiment_vote(rev)
    rows.append({"review": rev, "label": label,
                 **{f"p_{k}": round(v, 3) for k, v in probs.items()},
                 "vote_label": vote_label, "agreement": agreement, "votes": votes})

pd.DataFrame(rows)

,review,label,p_positive,p_negative,p_neutral,vote_label,agreement,votes
0,"Absolutely loved it, would buy again!",positive,1.0,0.000,0.000,positive,1.0,{'positive': 5}
1,The product arrived broken and support never a...,negative,0.0,1.000,0.000,negative,1.0,{'negative': 5}
2,It's a phone. It makes calls.,neutral,0.0,0.000,1.000,neutral,1.0,{'neutral': 5}
3,"Great screen, but the battery dies by lunchtime.",negative,0.0,0.962,0.037,negative,0.8,"{'neutral': 1, 'negative': 4}"


## 5. Text Completion
**Exercise:** Create a text completion application that generates text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)

In [11]:
def complete(prompt, **params):
    messages = [
        {"role": "system", "content": "Continue the user's text naturally. Do not repeat it and do not add any commentary."},
        {"role": "user", "content": prompt},
    ]
    r = client.chat.completions.create(model=MODEL, messages=messages, **params)
    return r.choices[0].message.content, r.choices[0].finish_reason

prompt = "Once upon a time, in a small mountain village, a clockmaker discovered that"

for params in [dict(temperature=0.2), dict(temperature=1.3), dict(max_tokens=25),
               dict(top_p=0.3), dict(frequency_penalty=1.8), dict(presence_penalty=1.8),
               dict(stop=["\n", "THE END"])]:
    text, reason = complete(prompt, max_tokens=params.pop("max_tokens", 120), **params)
    print(f"--- {params} ({reason}) ---\n{text}\n")

--- {'temperature': 0.2} (length) ---
his grandfather's old workshop held a secret. Among the dusty shelves and cobweb-covered tools, he found a peculiar clock unlike any he had ever seen. Its face was adorned with intricate carvings of celestial bodies, and its hands moved in a way that seemed almost alive. Intrigued, he decided to restore it, unaware that this clock held the power to manipulate time itself.

As he worked on the clock, he noticed that the air around him began to shimmer, and the sounds of the village faded into a distant hum. With each tick and tock, he felt a strange energy cours

--- {'temperature': 1.3} (length) ---
his grandfather's ancient workshop held not just the tools for crafting timepieces but also an old wooden chest. Curious about its contents, he carefully opened it and found a collection of unusual gears and materials bearing strange symbols. Among them was a delicate pocket watch, its surface shimmering under the sunlight, seemingly untouched by time.


In [17]:
completion_messages = [
    {"role": "system", "content": "Continue the user's text naturally. Do not repeat it and do not add commentary."},
    {"role": "user", "content": prompt},
]

for k in (1, 5):
    print(f"\n===== best_of={k} =====")
    best = chat_best_of(completion_messages, best_of=k, temperature=0.9,
                        max_tokens=100, stop=["\n\n", "THE END"])
    print(f"\nCHOSEN:\n{prompt} {best}")


===== best_of=1 =====
avg p = 45.2% | his grandfather's ancient clock had a secret mechanism hidden within its intricate gears. ...

CHOSEN:
Once upon a time, in a small mountain village, a clockmaker discovered that his grandfather's ancient clock had a secret mechanism hidden within its intricate gears. One fateful day, as he meticulously cleaned and oiled the timepiece, the clock struck midnight and a strange hum filled the air. Curious, he leaned in closer, and to his astonishment, a tiny door on the clock’s face swung open, revealing a shimmering key.



===== best_of=5 =====
avg p = 58.5% | his grandfather's old workshop held secrets beyond mere timekeeping. Among the dusty shelv...
avg p = 53.4% | his grandfather's old workshop held a secret he had never known. While rummaging through d...
avg p = 51.2% | his grandfather’s old clock had the unique ability to manipulate time. Each time he repair...
avg p = 50.3% | his old grandfather's clock was not just a timekeeper, but a gate

# BONUS: Google Vertex AI

## 1. Basic Conversation
**Exercise:** Create a basic chatbot using Google Vertex AI to answer questions about a given topic.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [14]:
# pip install google-genai
# Authenticate once in a terminal:  gcloud auth application-default login
from google import genai
from google.genai import types

gclient = genai.Client(vertexai=True, project="your-gcp-project-id", location="us-central1")
GEMINI = "gemini-2.5-flash"     # check the Vertex AI model list for what's currently available

def gemini(prompt, system=None, **config):
    cfg = types.GenerateContentConfig(system_instruction=system, **config)
    return gclient.models.generate_content(model=GEMINI, contents=prompt, config=cfg)

ImportError: cannot import name 'genai' from 'google' (unknown location)

## 2. Summarization
**Exercise:** Develop a script that summarizes long text inputs using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 3. Translation
**Exercise:** Create a tool that translates text from one language to another using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool using Google Vertex AI to determine the sentiment of a given text.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 5. Text Completion
**Exercise:** Develop a text completion application using Google Vertex AI to generate text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)